# Portfolio reproduction notes

This is a lightly sanitized archival research notebook. **It has not been fully refactored or rerun end-to-end as packaged.** Original notebook outputs were cleared for repository size and privacy.

Before execution: mount Google Drive, search for `YOUR_DRIVE_PROJECT_ROOT`, and replace that placeholder with the correct folder. **Review all cells before running**: some original cells can create train/validation/test splits, overwrite caches or save models. Avoid running the whole notebook automatically.

The research notebooks used the terms *without augmentation* and *with augmentation* for external images; the latter were actually produced by prompt-based generative image editing in CapCut. See [DATASET.md](../DATASET.md) and [LIMITATIONS.md](../LIMITATIONS.md).


**CONNECT AND LOAD DATASET**

In [ ]:
# MENGIMPOR MODUL DRIVE DARI GOOGLE COLAB

# Tujuannya untuk menghubungkan Google Colab dengan Google Drive
from google.colab import drive

# Melakukan mount Google Drive ke direktori /content/drive
# Setelah proses ini berhasil, file di Google Drive bisa diakses dari Colab
drive.mount('/content/drive')

In [ ]:
# MENYIMPAN PATH/FOLDER DATASET UTAMA

# Folder ini berisi gambar daun teh yang sudah dikelompokkan berdasarkan kelas
DATASET_PATH = "/content/drive/MyDrive/YOUR_DRIVE_PROJECT_ROOT/dataset"

# Menyimpan path/folder untuk dataset hasil split
# Biasanya folder ini digunakan untuk menyimpan data train, validation, dan test
SPLIT_PATH   = "/content/drive/MyDrive/YOUR_DRIVE_PROJECT_ROOT/dataset_split"

**PREPROCESSING**

In [ ]:
# PENGECEKAN FILE DUPLIKAT

import os
import hashlib
from collections import defaultdict

# Fungsi untuk menghitung hash MD5 dari sebuah file
# Hash ini digunakan untuk mendeteksi file yang isinya sama/duplikat
def get_md5(file_path):
    hash_md5 = hashlib.md5()

    # Membuka file dalam mode biner
    with open(file_path, "rb") as f:
        # Membaca file per chunk agar lebih efisien
        for chunk in iter(lambda: f.read(4096), b""):
            hash_md5.update(chunk)

    # Mengembalikan nilai hash MD5 dalam bentuk heksadesimal
    return hash_md5.hexdigest()

# Dictionary untuk menyimpan hash file sebagai key
# dan daftar path file yang memiliki hash sama sebagai value
hash_dict = defaultdict(list)

# Melakukan iterasi pada setiap folder kelas dalam dataset
for cls in os.listdir(DATASET_PATH):
    cls_path = os.path.join(DATASET_PATH, cls)

    # Melewati item yang bukan folder
    if not os.path.isdir(cls_path):
        continue

    # Melakukan iterasi pada setiap file gambar dalam folder kelas
    for img in os.listdir(cls_path):
        img_path = os.path.join(cls_path, img)
        try:
            # Menghitung hash file
            file_hash = get_md5(img_path)

            # Menyimpan path file berdasarkan nilai hash
            hash_dict[file_hash].append(img_path)
        except:
            # Jika file gagal dibaca, maka dilewati
            pass

# Mengambil hanya hash yang memiliki lebih dari satu file
# Artinya file-file tersebut terindikasi sebagai duplikat
duplicates = {k:v for k,v in hash_dict.items() if len(v) > 1}

# Menampilkan jumlah grup file duplikat
print(f"Jumlah file duplikat: {len(duplicates)}")

In [ ]:
# PENGECEKAN FILE GAMBAR RUSAK

import cv2

# List untuk menyimpan file yang rusak atau tidak bisa dibaca
broken_files = []

# Melakukan iterasi pada setiap folder kelas dalam dataset
for cls in os.listdir(DATASET_PATH):
    cls_path = os.path.join(DATASET_PATH, cls)

    # Melakukan iterasi pada setiap file gambar
    for img in os.listdir(cls_path):
        img_path = os.path.join(cls_path, img)

        # Membaca gambar menggunakan OpenCV
        image = cv2.imread(img_path)

        # Jika gambar tidak berhasil dibaca, maka dianggap file rusak
        if image is None:
            broken_files.append(img_path)

# Menampilkan jumlah file rusak dalam dataset
print("Jumlah file rusak:", len(broken_files))

In [ ]:
# MEMBAGI DATASET MENJADI TRAIN, VALIDATION, DAN TEST

import os, shutil, random

# Menentukan rasio pembagian dataset
# 80% untuk train, 10% untuk validation, dan 10% untuk test
train_ratio, val_ratio, test_ratio = 0.8, 0.1, 0.1

# Menetapkan seed random agar hasil split selalu sama saat kode dijalankan ulang
random.seed(42)

# Fungsi Mengecek apakah dataset sudah pernah di-split
def is_dataset_already_split(split_path):
    # Jika folder split belum ada, berarti dataset belum di-split
    if not os.path.exists(split_path):
        return False

    # Mengecek apakah folder train, val, dan test tersedia dan tidak kosong
    for split in ["train", "val", "test"]:
        split_dir = os.path.join(split_path, split)

        if not os.path.exists(split_dir):
            return False

        if len(os.listdir(split_dir)) == 0:
            return False

    return True

# Jika dataset sudah pernah di-split, proses dilewati
if is_dataset_already_split(SPLIT_PATH):
    print("✅ Dataset sudah di-split sebelumnya. Proses dilewati.")
else:
    print("🔄 Melakukan proses splitting dataset...")

    # Melakukan iterasi pada setiap folder kelas
    for cls in sorted(os.listdir(DATASET_PATH)):
        cls_path = os.path.join(DATASET_PATH, cls)

        # Melewati item yang bukan folder
        if not os.path.isdir(cls_path):
            continue

        # Mengambil seluruh gambar dalam kelas
        images = os.listdir(cls_path)

        # Mengacak urutan gambar agar pembagian dataset bersifat acak
        random.shuffle(images)

        # Menghitung jumlah total gambar dalam kelas
        n_total = len(images)

        # Menghitung jumlah data train dan validation
        n_train = int(train_ratio * n_total)
        n_val   = int(val_ratio * n_total)

        # Membagi daftar gambar ke dalam train, val, dan test
        splits = {
            "train": images[:n_train],
            "val"  : images[n_train:n_train+n_val],
            "test" : images[n_train+n_val:]
        }

        # Menyalin gambar ke folder split sesuai subset masing-masing
        for split, imgs in splits.items():
            split_dir = os.path.join(SPLIT_PATH, split, cls)

            # Membuat folder tujuan jika belum ada
            os.makedirs(split_dir, exist_ok=True)

            for img in imgs:
                src = os.path.join(cls_path, img)   # path file asal
                dst = os.path.join(split_dir, img)  # path file tujuan

                # Proteksi agar file yang sudah ada tidak disalin ulang
                if not os.path.exists(dst):
                    shutil.copy(src, dst)

    # Menampilkan informasi bahwa proses splitting selesai
    print("✅ Proses splitting dataset selesai.")

In [ ]:
# FUNGSI UNTUK MENGHITUNG JUMLAH GAMBAR PADA SETIAP KELAS

# di dalam folder train, val, atau test
def count_images_per_class(base_path):
    print(f"\nDataset: {base_path}")
    total = 0

    # Melakukan iterasi pada setiap folder kelas
    for cls in sorted(os.listdir(base_path)):
        cls_path = os.path.join(base_path, cls)

        # Memastikan item yang dibaca adalah folder kelas
        if os.path.isdir(cls_path):
            # Menghitung jumlah file gambar dalam folder kelas
            count = len(os.listdir(cls_path))
            total += count

            # Menampilkan jumlah gambar per kelas
            print(f"{cls:15s}: {count}")

    # Menampilkan jumlah total gambar dalam subset
    print(f"TOTAL: {total}")

# Menampilkan distribusi jumlah gambar pada data train
count_images_per_class(SPLIT_PATH + "/train")

# Menampilkan distribusi jumlah gambar pada data validation
count_images_per_class(SPLIT_PATH + "/val")

# Menampilkan distribusi jumlah gambar pada data test
count_images_per_class(SPLIT_PATH + "/test")

**EKSTRAKSI FITUR GLOBAL**

In [ ]:
# MEMBUAT DATA GENERATOR UNTUK INPUT VGG16

# Mengimpor ImageDataGenerator untuk membaca gambar dari folder
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Mengimpor fungsi preprocessing khusus untuk VGG16
# Fungsi ini menyesuaikan nilai piksel gambar agar sesuai dengan format input VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input

# Membuat objek data generator dengan preprocessing VGG16
datagen_vgg = ImageDataGenerator(preprocessing_function=preprocess_input)

# Membaca data train dari folder
# Semua gambar akan di-resize menjadi 224x224 piksel
# batch_size=16 berarti gambar diproses per 16 gambar dalam satu batch
# class_mode="categorical" digunakan untuk klasifikasi multikelas
# shuffle=False agar urutan data tetap dan sesuai dengan label aslinya
train_gen = datagen_vgg.flow_from_directory(
    SPLIT_PATH + "/train",
    target_size=(224,224),
    batch_size=16,
    class_mode="categorical",
    shuffle=False
)

# Membaca data validation dari folder
val_gen = datagen_vgg.flow_from_directory(
    SPLIT_PATH + "/val",
    target_size=(224,224),
    batch_size=16,
    class_mode="categorical",
    shuffle=False
)

# Membaca data test dari folder
test_gen = datagen_vgg.flow_from_directory(
    SPLIT_PATH + "/test",
    target_size=(224,224),
    batch_size=16,
    class_mode="categorical",
    shuffle=False
)

# Mengambil label kelas numerik dari masing-masing generator
y_train = train_gen.classes
y_val   = val_gen.classes
y_test  = test_gen.classes

# Mengambil nama-nama kelas berdasarkan urutan indeks kelas
class_names = list(train_gen.class_indices.keys())

In [ ]:
# MEMBANGUN Arsitektur VGG16 SEBAGAI EKSTRAKTOR FITUR GLOBAL

# Mengimpor model VGG16 pretrained dari Keras
from tensorflow.keras.applications import VGG16

# Mengimpor GlobalAveragePooling2D untuk merangkum feature map menjadi vektor fitur
from tensorflow.keras.layers import GlobalAveragePooling2D

# Mengimpor Model untuk membentuk model ekstraktor fitur baru
from tensorflow.keras.models import Model

# Mengimpor NumPy untuk manipulasi array
import numpy as np

# Memuat model VGG16 dengan bobot pretrained ImageNet
# include_top=False artinya layer klasifikasi bawaan VGG16 tidak digunakan
# input_shape=(224,224,3) menyesuaikan ukuran input citra RGB
base_model = VGG16(
    weights="imagenet",
    include_top=False,
    input_shape=(224,224,3)
)

# Menambahkan GlobalAveragePooling2D di atas output VGG16
# Tujuannya agar output feature map diubah menjadi vektor fitur yang ringkas
x = GlobalAveragePooling2D()(base_model.output)

# Membentuk model baru yang berfungsi sebagai ekstraktor fitur VGG16
vgg_extractor = Model(base_model.input, x)

# Membekukan seluruh layer model agar bobot tidak ikut dilatih ulang
# Model hanya digunakan sebagai feature extractor
for layer in vgg_extractor.layers:
    layer.trainable = False

# Menampilkan ringkasan arsitektur model
vgg_extractor.summary()

In [ ]:
# MENYIMPAN DAN MEMUAT FITUR HASIL EKSTRAKSI VGG16

import os
import numpy as np

# Menentukan folder penyimpanan fitur VGG16 di Google Drive
VGG_SAVE_PATH = "/content/drive/MyDrive/YOUR_DRIVE_PROJECT_ROOT/features_vgg16_lbp/"

# Membuat folder jika belum ada
os.makedirs(VGG_SAVE_PATH, exist_ok=True)

# Fungsi untuk memuat fitur VGG16 jika sudah ada
# atau mengekstraksi ulang jika file fitur belum tersedia
def load_or_extract_vgg(vgg_model, train_gen, val_gen, test_gen):
    # Menentukan nama file untuk menyimpan fitur train, val, dan test
    train_file = os.path.join(VGG_SAVE_PATH, "X_train_vgg.npy")
    val_file   = os.path.join(VGG_SAVE_PATH, "X_val_vgg.npy")
    test_file  = os.path.join(VGG_SAVE_PATH, "X_test_vgg.npy")

    # Jika semua file fitur sudah ada, langsung load dari Google Drive
    if all(os.path.exists(f) for f in [train_file, val_file, test_file]):
        print("✅ Load fitur VGG16 dari Google Drive")
        X_train_vgg = np.load(train_file)
        X_val_vgg   = np.load(val_file)
        X_test_vgg  = np.load(test_file)

    else:
        # Jika file fitur belum ada, lakukan ekstraksi fitur menggunakan VGG16
        print("🔄 Ekstraksi fitur VGG16 (sekali saja)...")
        X_train_vgg = vgg_model.predict(train_gen, verbose=1)
        X_val_vgg   = vgg_model.predict(val_gen, verbose=1)
        X_test_vgg  = vgg_model.predict(test_gen, verbose=1)

        # Menyimpan hasil ekstraksi fitur ke file .npy
        np.save(train_file, X_train_vgg)
        np.save(val_file, X_val_vgg)
        np.save(test_file, X_test_vgg)

    # Safety check:
    # Memastikan jumlah fitur sama dengan jumlah sampel pada masing-masing generator
    assert X_train_vgg.shape[0] == train_gen.samples
    assert X_val_vgg.shape[0]   == val_gen.samples
    assert X_test_vgg.shape[0]  == test_gen.samples

    # Mengembalikan fitur train, val, dan test
    return X_train_vgg, X_val_vgg, X_test_vgg

In [ ]:
# MENJALANKAN FUNGSI EKSTRAKSI ATAU PEMUATAN FITUR VGG16

# Menjalankan fungsi untuk memuat atau mengekstraksi fitur VGG16
X_train_vgg, X_val_vgg, X_test_vgg = load_or_extract_vgg(
    vgg_extractor,
    train_gen,
    val_gen,
    test_gen
)

# Menampilkan bentuk/dimensi fitur hasil ekstraksi
# Ini berguna untuk memastikan fitur berhasil dibuat dengan benar
print(X_train_vgg.shape, X_val_vgg.shape, X_test_vgg.shape)

**EKSTRAKSI FITUR LOKAL w LBP**

In [ ]:
# MEMBUAT DATA GENERATOR UNTUK EKSTRAKSI FITUR LBP

# Membuat data generator untuk ekstraksi fitur LBP
# rescale=1./255 digunakan untuk menormalkan nilai piksel ke rentang 0-1
datagen_lbp = ImageDataGenerator(rescale=1./255)

# Membaca data train dari folder
# Semua gambar di-resize menjadi 224x224 piksel
# class_mode=None karena pada tahap ini hanya gambar yang dibutuhkan, bukan label
# shuffle=False agar urutan data tetap konsisten
train_gen_lbp = datagen_lbp.flow_from_directory(
    SPLIT_PATH + "/train",
    target_size=(224,224),
    batch_size=16,
    class_mode=None,
    shuffle=False
)

# Membaca data validation dari folder
val_gen_lbp = datagen_lbp.flow_from_directory(
    SPLIT_PATH + "/val",
    target_size=(224,224),
    batch_size=16,
    class_mode=None,
    shuffle=False
)

# Membaca data test dari folder
test_gen_lbp = datagen_lbp.flow_from_directory(
    SPLIT_PATH + "/test",
    target_size=(224,224),
    batch_size=16,
    class_mode=None,
    shuffle=False
)

In [ ]:
# MEMBUAT FUNGSI EKSTRAKSI FITUR LOKAL DENGAN LBP PER BATCH DAN PER GENERATOR

import cv2
import numpy as np

# Mengimpor fungsi local_binary_pattern dari skimage
from skimage.feature import local_binary_pattern

# Fungsi untuk mengekstraksi fitur LBP dari satu batch gambar
def extract_lbp_batch(batch_images, radius=1, n_points=8):
    features = []

    # Jumlah bin histogram untuk metode uniform
    n_bins = n_points + 2

    # Melakukan iterasi pada setiap gambar dalam batch
    for img in batch_images:
        # Mengembalikan skala piksel dari 0-1 menjadi 0-255
        # lalu mengubah tipe data menjadi uint8
        img = (img * 255).astype(np.uint8)

        # Memastikan gambar dalam bentuk grayscale
        # Jika gambar masih RGB, ubah menjadi grayscale
        if img.ndim == 3 and img.shape[2] == 3:
            gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
        else:
            gray = img

        # Menghitung representasi Local Binary Pattern (LBP)
        lbp = local_binary_pattern(
            gray,
            n_points,      # jumlah titik tetangga
            radius,        # radius tetangga
            method="uniform"
        )

        # Mengubah hasil LBP menjadi histogram fitur
        hist, _ = np.histogram(
            lbp.ravel(),
            bins=n_bins,
            range=(0, n_bins),
            density=True   # histogram dinormalisasi
        )

        # Menyimpan histogram sebagai fitur gambar
        features.append(hist)

    # Mengembalikan semua fitur dalam bentuk array NumPy
    return np.array(features)

# Fungsi untuk mengekstraksi fitur LBP dari seluruh generator
def extract_lbp_generator(generator):
    all_feat = []

    # Menghitung jumlah batch yang harus diproses
    steps = int(np.ceil(generator.samples / generator.batch_size))

    # Melakukan iterasi sebanyak jumlah batch
    for _ in range(steps):
        batch = next(generator)

        # Jika output generator berupa tuple, ambil bagian gambar saja
        # Jika langsung berupa array gambar, gunakan langsung
        batch_images = batch[0] if isinstance(batch, tuple) else batch

        # Mengekstraksi fitur LBP dari batch gambar
        lbp_feat = extract_lbp_batch(batch_images)

        # Menyimpan hasil fitur batch
        all_feat.append(lbp_feat)

    # Menggabungkan seluruh fitur batch menjadi satu array
    return np.vstack(all_feat)

In [ ]:
# MENYIMPAN DAN MEMUAT FITUR HASIL EKSTRAKSI LBP

import os
import numpy as np

# Menentukan folder penyimpanan fitur LBP di Google Drive
LBP_SAVE_PATH = "/content/drive/MyDrive/YOUR_DRIVE_PROJECT_ROOT/features_vgg16_lbp/"

# Membuat folder jika belum ada
os.makedirs(LBP_SAVE_PATH, exist_ok=True)

# Fungsi untuk memuat fitur LBP jika sudah tersedia
# atau mengekstraksi ulang jika file belum ada
def load_or_extract_lbp(train_gen, val_gen, test_gen):
    # Menentukan nama file penyimpanan fitur
    train_file = os.path.join(LBP_SAVE_PATH, "X_train_lbp.npy")
    val_file   = os.path.join(LBP_SAVE_PATH, "X_val_lbp.npy")
    test_file  = os.path.join(LBP_SAVE_PATH, "X_test_lbp.npy")

    # Jika semua file fitur sudah ada, langsung load dari Google Drive
    if all(os.path.exists(f) for f in [train_file, val_file, test_file]):
        print("✅ Load fitur LBP dari Google Drive")
        X_train = np.load(train_file)
        X_val   = np.load(val_file)
        X_test  = np.load(test_file)

    else:
        # Jika belum ada, lakukan ekstraksi fitur LBP
        print("🔄 Ekstraksi fitur LBP (sekali saja)...")
        X_train = extract_lbp_generator(train_gen)
        X_val   = extract_lbp_generator(val_gen)
        X_test  = extract_lbp_generator(test_gen)

        # Menyimpan hasil fitur ke file .npy
        np.save(train_file, X_train)
        np.save(val_file, X_val)
        np.save(test_file, X_test)

    # Safety check:
    # Memastikan jumlah fitur sama dengan jumlah sampel pada generator
    assert X_train.shape[0] == train_gen.samples
    assert X_val.shape[0]   == val_gen.samples
    assert X_test.shape[0]  == test_gen.samples

    # Mengembalikan fitur train, val, dan test
    return X_train, X_val, X_test

In [ ]:
# MENJALANKAN PROSES LOAD ATAU EKSTRAKSI FITUR LBP

# Menjalankan fungsi untuk memuat atau mengekstraksi fitur LBP
X_train_lbp, X_val_lbp, X_test_lbp = load_or_extract_lbp(
    train_gen_lbp,
    val_gen_lbp,
    test_gen_lbp
)

# Menampilkan bentuk/dimensi fitur hasil ekstraksi LBP
print(X_train_lbp.shape, X_val_lbp.shape, X_test_lbp.shape)

**FITURE CONCANTENATION VGG16xLBP & NORMALISASI**

In [ ]:
# MENGIMPOR LIBRARY DAN MENYIAPKAN FOLDER PENYIMPANAN FITUR FUSI SERTA MEMBUAT FUNGSI UNTUK MEMUAT ATAU MEMBANGUN FITUR HASIL FUSI

# Mengimpor StandardScaler untuk melakukan standardisasi fitur
from sklearn.preprocessing import StandardScaler

# Mengimpor joblib untuk menyimpan dan memuat objek scaler
import joblib

# Mengimpor library yang dibutuhkan untuk manipulasi file dan array
import os
import numpy as np

# Menentukan folder penyimpanan fitur hasil fusi
FUSED_SAVE_PATH = "/content/drive/MyDrive/YOUR_DRIVE_PROJECT_ROOT/features_vgg16_lbp/"

# Membuat folder jika belum ada
os.makedirs(FUSED_SAVE_PATH, exist_ok=True)

# Fungsi untuk memuat fitur hasil fusi jika sudah tersedia
# atau membangun ulang jika file belum ada
def load_or_build_fused_features(
    X_train_vgg, X_val_vgg, X_test_vgg,
    X_train_lbp, X_val_lbp, X_test_lbp
):
    # Menentukan nama file penyimpanan fitur hasil fusi dan scaler
    train_file  = os.path.join(FUSED_SAVE_PATH, "X_train_fused_scaled.npy")
    val_file    = os.path.join(FUSED_SAVE_PATH, "X_val_fused_scaled.npy")
    test_file   = os.path.join(FUSED_SAVE_PATH, "X_test_fused_scaled.npy")
    scaler_file = os.path.join(FUSED_SAVE_PATH, "scaler.save")

    # Jika semua file sudah ada, langsung load dari Google Drive
    if all(os.path.exists(f) for f in [train_file, val_file, test_file, scaler_file]):
        print("✅ Load fitur hasil fusi + scaler")
        scaler = joblib.load(scaler_file)
        return (
            np.load(train_file),
            np.load(val_file),
            np.load(test_file),
            scaler
        )

    print("🔄 Concatenate + Scaling fitur (sekali saja)...")

    # Menggabungkan fitur VGG16 dan fitur LBP secara horizontal
    # axis=1 artinya penggabungan dilakukan pada kolom fitur
    X_train_fused = np.concatenate([X_train_vgg, X_train_lbp], axis=1)
    X_val_fused   = np.concatenate([X_val_vgg, X_val_lbp], axis=1)
    X_test_fused  = np.concatenate([X_test_vgg, X_test_lbp], axis=1)

    # Melakukan standardisasi fitur
    # Scaler hanya di-fit pada data train untuk menghindari data leakage
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_fused)
    X_val_scaled   = scaler.transform(X_val_fused)
    X_test_scaled  = scaler.transform(X_test_fused)

    # Menyimpan fitur hasil fusi yang sudah diskalakan
    np.save(train_file, X_train_scaled)
    np.save(val_file, X_val_scaled)
    np.save(test_file, X_test_scaled)

    # Menyimpan objek scaler agar bisa digunakan kembali
    joblib.dump(scaler, scaler_file)

    # Mengembalikan fitur train, val, test yang sudah diskalakan beserta scaler
    return X_train_scaled, X_val_scaled, X_test_scaled, scaler

In [ ]:
# MENJALANKAN FUNGSI PEMBENTUKAN FITUR HASIL FUSI

# Menjalankan fungsi untuk memuat atau membangun fitur hasil fusi
X_train_scaled, X_val_scaled, X_test_scaled, scaler = load_or_build_fused_features(
    X_train_vgg, X_val_vgg, X_test_vgg,
    X_train_lbp, X_val_lbp, X_test_lbp
)

# Menampilkan dimensi fitur
print(X_train_scaled.shape, X_val_scaled.shape, X_test_scaled.shape)

**MODEL KLASIFIKASI SVM**

In [ ]:
# TUNING SVM MENGGUNAKAN DATA VALIDASI

from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import pandas as pd
import numpy as np

# Daftar kandidat hyperparameter
param_grid = [
    {"kernel": "rbf",    "C": 0.1,  "gamma": "scale"},
    {"kernel": "rbf",    "C": 1,    "gamma": "scale"},
    {"kernel": "rbf",    "C": 10,   "gamma": "scale"},
    {"kernel": "rbf",    "C": 100,  "gamma": "scale"},
    {"kernel": "linear", "C": 0.1},
    {"kernel": "linear", "C": 1},
    {"kernel": "linear", "C": 10},
    {"kernel": "linear", "C": 100},
]

results = []
best_model = None
best_params = None
best_val_acc = -1
best_val_f1  = -1

for params in param_grid:
    model = SVC(
        **params,
        probability=True,
        random_state=42
    )

    # Train hanya pada data train
    model.fit(X_train_scaled, y_train)

    # Prediksi pada data validasi
    y_val_pred = model.predict(X_val_scaled)

    # Hitung metrik
    val_acc = accuracy_score(y_val, y_val_pred)
    val_f1  = f1_score(y_val, y_val_pred, average="macro")

    results.append({
        "params": params,
        "val_accuracy": val_acc,
        "val_f1_macro": val_f1
    })

    print(f"Params: {params} | Val Accuracy: {val_acc:.4f} | Val F1-macro: {val_f1:.4f}")

    # Pilih model terbaik:
    # prioritas 1 = accuracy validasi tertinggi
    # prioritas 2 = kalau accuracy sama, pilih F1-macro lebih tinggi
    if (val_acc > best_val_acc) or (val_acc == best_val_acc and val_f1 > best_val_f1):
        best_val_acc = val_acc
        best_val_f1  = val_f1
        best_model = model
        best_params = params

In [ ]:
# EVALUASI MODEL TERBAIK PADA DATA VALIDASI

y_val_pred_best = best_model.predict(X_val_scaled)

print("=== CLASSIFICATION REPORT (VALIDATION) ===")
print(classification_report(y_val, y_val_pred_best, target_names=class_names))

print("=== CONFUSION MATRIX (VALIDATION) ===")
print(confusion_matrix(y_val, y_val_pred_best))

In [ ]:
# TRAIN FINAL MODEL DENGAN TRAIN + VALIDATION

from sklearn.preprocessing import StandardScaler
import joblib
import os

# Gabungkan fitur mentah (sebelum scaling) untuk train + val
X_trainval_fused = np.concatenate([
    np.concatenate([X_train_vgg, X_train_lbp], axis=1),
    np.concatenate([X_val_vgg,   X_val_lbp],   axis=1)
], axis=0)

y_trainval = np.concatenate([y_train, y_val], axis=0)

# Fitur test mentah
X_test_fused = np.concatenate([X_test_vgg, X_test_lbp], axis=1)

# Fit scaler BARU pada gabungan train+val
final_scaler = StandardScaler()
X_trainval_scaled = final_scaler.fit_transform(X_trainval_fused)
X_test_final_scaled = final_scaler.transform(X_test_fused)

# Bangun model final dengan hyperparameter terbaik
final_svm_model = SVC(
    **best_params,
    probability=True,
    random_state=42
)

# Latih model final
final_svm_model.fit(X_trainval_scaled, y_trainval)

In [ ]:
# SIMPAN MODEL FINAL + SCALER FINAL

MODEL_SAVE_PATH = "/content/drive/MyDrive/YOUR_DRIVE_PROJECT_ROOT/features_vgg16_lbp/"
os.makedirs(MODEL_SAVE_PATH, exist_ok=True)

joblib.dump(final_svm_model, os.path.join(MODEL_SAVE_PATH, "svm_model_final.joblib"))
joblib.dump(final_scaler, os.path.join(MODEL_SAVE_PATH, "scaler_final.joblib"))

print("✅ Model final dan scaler final berhasil disimpan.")

print(type(final_svm_model))
print(type(final_scaler))
print(X_test_final_scaled.shape)

**PENGUJIAN DATA INTERNAL**

In [ ]:
# PENGUJIAN DATA INTERNAL

from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import seaborn as sns
import matplotlib.pyplot as plt
import cv2
import numpy as np
import random

In [ ]:
# HASIL UJI AKURASI

# Prediksi pada data test menggunakan model final
y_test_pred = final_svm_model.predict(X_test_final_scaled)

# Akurasi test
test_accuracy = accuracy_score(y_test, y_test_pred)
print(f"Akurasi Pengujian Data Internal (VGG16 + LBP): {test_accuracy:.2f}")

In [ ]:
# HASIL CLASSIFICATION REPORT

print("\nClassification Report Pengujian Data Internal (VGG16 + LBP):")
print(classification_report(
    y_test,
    y_test_pred,
    target_names=class_names
))

In [ ]:
# HASIL CONFUSION MATRIX

cm = confusion_matrix(y_test, y_test_pred)

plt.figure(figsize=(8,6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    xticklabels=class_names,
    yticklabels=class_names,
    cmap="Blues"
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix Pengujian Data Internal (VGG16 + LBP)")
plt.show()

In [ ]:
# HASIL UJI KLASIFIKASI GAMBAR

import matplotlib.pyplot as plt
import cv2
import numpy as np

def show_predictions_per_class_lbp(
    filepaths,
    y_true,
    y_pred,
    y_proba,
    class_names,
    samples_per_class=5,
    title_prefix="Hasil Uji Klasifikasi"
):
    """
    Menampilkan hasil prediksi secara terstruktur per kelas asli.
    Dari setiap kelas diambil beberapa gambar, lalu ditampilkan:
    - gambar
    - label asli
    - label prediksi
    - confidence
    """

    # Loop untuk setiap kelas
    for class_idx, class_name in enumerate(class_names):
        # Cari semua indeks yang termasuk kelas tersebut
        class_indices = np.where(y_true == class_idx)[0]

        if len(class_indices) == 0:
            print(f"Tidak ada data untuk kelas: {class_name}")
            continue

        # Ambil maksimal 5 gambar pertama dari kelas tersebut
        selected_indices = class_indices[:samples_per_class]

        # Buat figure untuk 1 kelas
        plt.figure(figsize=(12, 4))
        plt.suptitle(f"{title_prefix} - Kelas: {class_name}", fontsize=14)

        for i, idx in enumerate(selected_indices):
            plt.subplot(1, samples_per_class, i + 1)

            # Baca gambar
            img_path = filepaths[idx]
            img = cv2.imread(img_path)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

            # Ambil label asli, prediksi, dan confidence
            true_label = class_names[y_true[idx]]
            pred_label = class_names[y_pred[idx]]

            # Tampilkan gambar
            plt.imshow(img)
            plt.title(
                f"Asli: {true_label}\n"
                f"Prediksi: {pred_label}\n",
                fontsize=10
            )
            plt.axis("off")

        plt.tight_layout()
        plt.show()

# Prediksi data test internal menggunakan final model
y_test_pred = final_svm_model.predict(X_test_final_scaled)
y_test_proba = final_svm_model.predict_proba(X_test_final_scaled)

# Tampilkan 5 gambar per kelas
show_predictions_per_class_lbp(
    filepaths=test_gen.filepaths,
    y_true=y_test,
    y_pred=y_test_pred,
    y_proba=y_test_proba,
    class_names=class_names,
    samples_per_class=5,
    title_prefix="Hasil Uji Klasifikasi Pengujian Data Internal (VGG16 + LBP)"
)

**PENGUJIAN DATA EKSTERNAL TANPA AUGMENTASI**

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import seaborn as sns
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.vgg16 import preprocess_input

from sklearn.metrics import classification_report, accuracy_score

# 1) KONFIGURASI

EXTERNAL_PATH = "/content/drive/MyDrive/YOUR_DRIVE_PROJECT_ROOT/data_uji_tanpa_augmentasi/"

# Urutan kelas utama sesuai penelitian/model
class_names = [
    'algal_spot',
    'brown_blight',
    'gray_blight',
    'healthy',
    'helopeltis',
    'red_spot'
]

class_to_idx = {
    'algal_spot': 0,
    'brown_blight': 1,
    'gray_blight': 2,
    'healthy': 3,
    'helopeltis': 4,
    'red_spot': 5
}

# Label lengkap 6 kelas
all_labels = np.arange(len(class_names))

# Kelas yang digunakan pada data eksternal asli
# gray_blight dan red_spot tidak digunakan
active_labels = [0, 1, 3, 4]
active_class_names = [class_names[i] for i in active_labels]

# Label yang tidak digunakan pada data eksternal asli
unused_labels = [i for i in range(len(class_names)) if i not in active_labels]

print("Kelas utama model:")
for idx, name in enumerate(class_names):
    print(f"{idx} = {name}")

print("\nKelas aktif data eksternal asli:")
for label in active_labels:
    print(f"{label} = {class_names[label]}")

print("\nKelas tidak digunakan pada data eksternal asli:")
for label in unused_labels:
    print(f"{label} = {class_names[label]}")

In [ ]:
# 2) MEMBACA FILE GAMBAR DAN MEMBUAT LABEL ASLI SECARA MANUAL

external_dir = Path(EXTERNAL_PATH)

image_extensions = [
    "*.jpg", "*.jpeg", "*.png", "*.bmp", "*.webp",
    "*.JPG", "*.JPEG", "*.PNG", "*.BMP", "*.WEBP"
]

image_paths = []
y_ext_true = []

for class_name in class_names:
    class_dir = external_dir / class_name

    if not class_dir.exists():
        print(f"Folder tidak ditemukan, dilewati: {class_name}")
        continue

    class_files = []

    for ext in image_extensions:
        class_files.extend(sorted(class_dir.glob(ext)))

    for img_path in sorted(class_files):
        image_paths.append(str(img_path))
        y_ext_true.append(class_to_idx[class_name])

y_ext_true = np.array(y_ext_true)

print("\nJumlah gambar:", len(image_paths))
print("Label unik pada y_ext_true:", np.unique(y_ext_true))
print("Jumlah tiap label:", np.bincount(y_ext_true, minlength=len(class_names)))

print("\nDistribusi data eksternal asli:")
for idx, name in enumerate(class_names):
    jumlah = np.sum(y_ext_true == idx)
    print(f"{name}: {jumlah}")

In [ ]:
# 3) MEMBUAT GENERATOR BERDASARKAN image_paths

df_ext = pd.DataFrame({
    "filepath": image_paths
})

BATCH_SIZE = 16
IMG_SIZE = (224, 224)

# Generator untuk VGG16
external_datagen_vgg = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

external_gen_vgg = external_datagen_vgg.flow_from_dataframe(
    dataframe=df_ext,
    x_col="filepath",
    y_col=None,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode=None,
    shuffle=False
)

# Generator untuk LBP
external_datagen_lbp = ImageDataGenerator(
    rescale=1./255
)

external_gen_lbp = external_datagen_lbp.flow_from_dataframe(
    dataframe=df_ext,
    x_col="filepath",
    y_col=None,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode=None,
    shuffle=False
)

In [ ]:
# 4) EKSTRAKSI FITUR VGG16 DAN LBP

external_gen_vgg.reset()
external_gen_lbp.reset()

# Ekstraksi fitur VGG16
X_ext_vgg = vgg_extractor.predict(
    external_gen_vgg,
    verbose=1
)

# Jika output VGG masih berdimensi lebih dari 2, ubah menjadi 2D
if len(X_ext_vgg.shape) > 2:
    X_ext_vgg = X_ext_vgg.reshape(X_ext_vgg.shape[0], -1)

# Ekstraksi fitur LBP
X_ext_lbp = extract_lbp_generator(external_gen_lbp)

In [ ]:
# 5) FUSI FITUR + SCALING + PREDIKSI

# Fusi fitur VGG16 + LBP
X_ext_fused = np.concatenate(
    [X_ext_vgg, X_ext_lbp],
    axis=1
)

# Scaling menggunakan scaler final dari training VGG16 + LBP
X_ext_final_scaled = final_scaler.transform(X_ext_fused)

# Prediksi menggunakan model SVM final
y_ext_pred = final_svm_model.predict(X_ext_final_scaled)

# Probabilitas prediksi
y_ext_proba = final_svm_model.predict_proba(X_ext_final_scaled)

In [ ]:
# HASIL UJI AKURASI

ext_accuracy = accuracy_score(y_ext_true, y_ext_pred)
print(f"Akurasi Pengujian Data Eksternal tanpa Augmentasi (VGG16 + LBP): {ext_accuracy:.2f}")

In [ ]:
# HASIL CLASSIFICATION REPORT

print("\nClassification Report Pengujian Data Eksternal tanpa Augmentasi (VGG16 + LBP):")
print(classification_report(
    y_ext_true,
    y_ext_pred,
    labels=active_labels,
    target_names=active_class_names,
    digits=2,
    zero_division=0
))

In [ ]:
# HASIL CONFUSION MATRIX

cm = confusion_matrix(
    y_ext_true,
    y_ext_pred,
    labels=active_labels
)

plt.figure(figsize=(7, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=active_class_names,
    yticklabels=active_class_names
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix Pengujian Data Eksternal tanpa Augmentasi (VGG16 + LBP):")
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# HASIL UJI KLASIFIKASI GAMBAR

import matplotlib.pyplot as plt
import cv2
import numpy as np

def show_predictions_per_class_lbp(
    filepaths,
    y_true,
    y_pred,
    y_proba,
    class_names,
    active_labels,
    model_classes=None,
    samples_per_class=5,
    title_prefix="Hasil Uji Klasifikasi Pengujian Data Eksternal tanpa Augmentasi (VGG16 + LBP)"
):
    """
    Menampilkan hasil prediksi secara terstruktur per kelas asli.
    Dari setiap kelas diambil beberapa gambar, lalu ditampilkan:
    - gambar
    - label asli
    - label prediksi
    - confidence
    """

    # Loop untuk setiap kelas
    for class_idx in active_labels:
        class_name = class_names[class_idx]

        # Cari semua indeks yang termasuk kelas tersebut
        class_indices = np.where(y_true == class_idx)[0]

        if len(class_indices) == 0:
            print(f"Tidak ada data untuk kelas: {class_name}")
            continue

        # Ambil maksimal 5 gambar pertama dari kelas tersebut
        selected_indices = class_indices[:samples_per_class]

        # Buat figure untuk 1 kelas
        plt.figure(figsize=(12, 4))
        plt.suptitle(f"{title_prefix} - Kelas: {class_name}", fontsize=14)

        for i, idx in enumerate(selected_indices):
            plt.subplot(1, samples_per_class, i + 1)

            # Baca gambar
            img_path = filepaths[idx]
            img = cv2.imread(img_path)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

            # Ambil label asli, prediksi, dan confidence
            true_label = class_names[y_true[idx]]
            pred_label = class_names[y_pred[idx]]

            # Tampilkan gambar
            plt.imshow(img)
            plt.title(
                f"Asli: {true_label}\n"
                f"Pred: {pred_label}\n",
                fontsize=10
            )
            plt.axis("off")

        plt.tight_layout()
        plt.show()

# Prediksi data eksternal menggunakan final model
y_ext_pred = final_svm_model.predict(X_ext_final_scaled)
y_ext_proba = final_svm_model.predict_proba(X_ext_final_scaled)

# Tampilkan 5 gambar per kelas
show_predictions_per_class_lbp(
    filepaths=image_paths,
    y_true=y_ext_true,
    y_pred=y_ext_pred,
    y_proba=y_ext_proba,
    class_names=class_names,
    active_labels=active_labels,
    model_classes=final_svm_model.classes_,
    samples_per_class=5,
    title_prefix="Hasil Uji Klasifikasi Pengujian Data Eksternal tanpa Augmentasi (VGG16 + LBP)"
)

**PENGUJIAN DATA EKSTERNAL DENGAN AUGMENTASI**

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import seaborn as sns
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.vgg16 import preprocess_input

from sklearn.metrics import classification_report, accuracy_score

# 1) KONFIGURASI

EXTERNAL_PATH = "/content/drive/MyDrive/YOUR_DRIVE_PROJECT_ROOT/data_uji_dengan_augmentasi/"

# Urutan kelas utama sesuai penelitian/model
class_names = [
    'algal_spot',
    'brown_blight',
    'gray_blight',
    'healthy',
    'helopeltis',
    'red_spot'
]

class_to_idx = {
    'algal_spot': 0,
    'brown_blight': 1,
    'gray_blight': 2,
    'healthy': 3,
    'helopeltis': 4,
    'red_spot': 5
}

# Label lengkap 6 kelas
all_labels = np.arange(len(class_names))

# Kelas yang digunakan pada data eksternal asli
# gray_blight dan red_spot tidak digunakan
active_labels = [0, 1, 3, 4]
active_class_names = [class_names[i] for i in active_labels]

# Label yang tidak digunakan pada data eksternal asli
unused_labels = [i for i in range(len(class_names)) if i not in active_labels]

print("Kelas utama model:")
for idx, name in enumerate(class_names):
    print(f"{idx} = {name}")

print("\nKelas aktif data eksternal asli:")
for label in active_labels:
    print(f"{label} = {class_names[label]}")

print("\nKelas tidak digunakan pada data eksternal asli:")
for label in unused_labels:
    print(f"{label} = {class_names[label]}")

In [ ]:
# 2) MEMBACA FILE GAMBAR DAN MEMBUAT LABEL ASLI SECARA MANUAL

external_dir = Path(EXTERNAL_PATH)

image_extensions = [
    "*.jpg", "*.jpeg", "*.png", "*.bmp", "*.webp",
    "*.JPG", "*.JPEG", "*.PNG", "*.BMP", "*.WEBP"
]

image_paths = []
y_ext_true = []

for class_name in class_names:
    class_dir = external_dir / class_name

    if not class_dir.exists():
        print(f"Folder tidak ditemukan, dilewati: {class_name}")
        continue

    class_files = []

    for ext in image_extensions:
        class_files.extend(sorted(class_dir.glob(ext)))

    for img_path in sorted(class_files):
        image_paths.append(str(img_path))
        y_ext_true.append(class_to_idx[class_name])

y_ext_true = np.array(y_ext_true)

print("\nJumlah gambar:", len(image_paths))
print("Label unik pada y_ext_true:", np.unique(y_ext_true))
print("Jumlah tiap label:", np.bincount(y_ext_true, minlength=len(class_names)))

print("\nDistribusi data eksternal asli:")
for idx, name in enumerate(class_names):
    jumlah = np.sum(y_ext_true == idx)
    print(f"{name}: {jumlah}")

In [ ]:
# 3) MEMBUAT GENERATOR BERDASARKAN image_paths

df_ext = pd.DataFrame({
    "filepath": image_paths
})

BATCH_SIZE = 16
IMG_SIZE = (224, 224)

# Generator untuk VGG16
external_datagen_vgg = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

external_gen_vgg = external_datagen_vgg.flow_from_dataframe(
    dataframe=df_ext,
    x_col="filepath",
    y_col=None,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode=None,
    shuffle=False
)

# Generator untuk LBP
external_datagen_lbp = ImageDataGenerator(
    rescale=1./255
)

external_gen_lbp = external_datagen_lbp.flow_from_dataframe(
    dataframe=df_ext,
    x_col="filepath",
    y_col=None,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode=None,
    shuffle=False
)

In [ ]:
# 4) EKSTRAKSI FITUR VGG16 DAN LBP

external_gen_vgg.reset()
external_gen_lbp.reset()

# Ekstraksi fitur VGG16
X_ext_vgg = vgg_extractor.predict(
    external_gen_vgg,
    verbose=1
)

# Jika output VGG masih berdimensi lebih dari 2, ubah menjadi 2D
if len(X_ext_vgg.shape) > 2:
    X_ext_vgg = X_ext_vgg.reshape(X_ext_vgg.shape[0], -1)

# Ekstraksi fitur LBP
X_ext_lbp = extract_lbp_generator(external_gen_lbp)

In [ ]:
# 5) FUSI FITUR + SCALING + PREDIKSI

# Fusi fitur VGG16 + LBP
X_ext_fused = np.concatenate(
    [X_ext_vgg, X_ext_lbp],
    axis=1
)

# Scaling menggunakan scaler final dari training VGG16 + LBP
X_ext_final_scaled = final_scaler.transform(X_ext_fused)

# Prediksi menggunakan model SVM final
y_ext_pred = final_svm_model.predict(X_ext_final_scaled)

# Probabilitas prediksi
y_ext_proba = final_svm_model.predict_proba(X_ext_final_scaled)

In [ ]:
# HASIL UJI AKURASI

ext_accuracy = accuracy_score(y_ext_true, y_ext_pred)
print(f"Akurasi Pengujian Data Eksternal dengan Augmentasi (VGG16 + LBP): {ext_accuracy:.2f}")

In [ ]:
# HASIL CLASSIFICATION REPORT

print("\nClassification Report Pengujian Data Eksternal dengan Augmentasi (VGG16 + LBP):")
print(classification_report(
    y_ext_true,
    y_ext_pred,
    labels=active_labels,
    target_names=active_class_names,
    digits=2,
    zero_division=0
))

In [ ]:
# HASIL CONFUSION MATRIX

cm = confusion_matrix(
    y_ext_true,
    y_ext_pred,
    labels=active_labels
)

plt.figure(figsize=(7, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=active_class_names,
    yticklabels=active_class_names
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix Pengujian Data Eksternal dengan Augmentasi (VGG16 + LBP)")
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# HASIL UJI KLASIFIKASI GAMBAR

import matplotlib.pyplot as plt
import cv2
import numpy as np

def show_predictions_per_class_lbp(
    filepaths,
    y_true,
    y_pred,
    y_proba,
    class_names,
    active_labels,
    model_classes=None,
    samples_per_class=5,
    title_prefix="Hasil Uji Klasifikasi Pengujian Data Eksternal tanpa Augmentasi (VGG16 + LBP)"
):
    """
    Menampilkan hasil prediksi secara terstruktur per kelas asli.
    Dari setiap kelas diambil beberapa gambar, lalu ditampilkan:
    - gambar
    - label asli
    - label prediksi
    - confidence
    """

    # Loop untuk setiap kelas
    for class_idx in active_labels:
        class_name = class_names[class_idx]

        # Cari semua indeks yang termasuk kelas tersebut
        class_indices = np.where(y_true == class_idx)[0]

        if len(class_indices) == 0:
            print(f"Tidak ada data untuk kelas: {class_name}")
            continue

        # Ambil maksimal 5 gambar pertama dari kelas tersebut
        selected_indices = class_indices[:samples_per_class]

        # Buat figure untuk 1 kelas
        plt.figure(figsize=(12, 4))
        plt.suptitle(f"{title_prefix} - Kelas: {class_name}", fontsize=14)

        for i, idx in enumerate(selected_indices):
            plt.subplot(1, samples_per_class, i + 1)

            # Baca gambar
            img_path = filepaths[idx]
            img = cv2.imread(img_path)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

            # Ambil label asli, prediksi, dan confidence
            true_label = class_names[y_true[idx]]
            pred_label = class_names[y_pred[idx]]

            # Tampilkan gambar
            plt.imshow(img)
            plt.title(
                f"Asli: {true_label}\n"
                f"Pred: {pred_label}\n",
                fontsize=10
            )
            plt.axis("off")

        plt.tight_layout()
        plt.show()

# Prediksi data eksternal menggunakan final model
y_ext_pred = final_svm_model.predict(X_ext_final_scaled)
y_ext_proba = final_svm_model.predict_proba(X_ext_final_scaled)

# Tampilkan 5 gambar per kelas
show_predictions_per_class_lbp(
    filepaths=image_paths,
    y_true=y_ext_true,
    y_pred=y_ext_pred,
    y_proba=y_ext_proba,
    class_names=class_names,
    active_labels=active_labels,
    model_classes=final_svm_model.classes_,
    samples_per_class=5,
    title_prefix="Hasil Uji Klasifikasi Pengujian Data Eksternal dengan Augmentasi (VGG16 + LBP)"
)